В этом ноутбуке мы будем работать над моделью предсказания движеняи цены акций по новостям, чтобы выявить важность признаков, а также актуальность бизнес-задачи.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("finhub+theguardian_raw_7.5k.csv", index_col=0)
df = df.dropna(subset=["fields.trailText"])
df.shape

(7353, 48)

В нашем датасете есть новости для которых есть краткая сводка, а есть новости для которых есть и краткая сводка и полный текст статьи. Для удобства будет обозначать их как short и full (или же s и f)

In [3]:
df["fields.trailText"]

0       Google, Amazon, Apple and Samsung control oper...
1       CMA intends to force tech firms to make change...
2       National Association of Cider Makers says warm...
3       First penalties under landmark Digital Markets...
4       CMA examining impact of tech firms’ operating ...
                              ...                        
7374     Cisco, 3M share gains lead Dow's 160-point climb
7375    A key trading signal flashed for Caterpillar s...
7376    Caterpillar and Walmart have been highlighted ...
7377    Atlas Energy Solutions Inc. (NYSE:AESI) (&#34;...
7378    Macroeconomic and sector-specific issues vary ...
Name: fields.trailText, Length: 7353, dtype: str

In [4]:
df["fields.bodyText"]

0       The world’s largest broadcasters have pushed f...
1       A UK watchdog has said it intends to take acti...
2       “If you love cider, this is cider to the power...
3       The European Commission has fined Apple €500m ...
4       The UK’s competition watchdog has launched inv...
                              ...                        
7374                                                  NaN
7375                                                  NaN
7376                                                  NaN
7377                                                  NaN
7378                                                  NaN
Name: fields.bodyText, Length: 7353, dtype: str

Ячейками ниже преобразуем наши текста в эмбеддинги. Код ниже был запущен в ноутубке на kaggle, поскольку на CPU данная операция работает очень медленно. Код был перенесен сюда и закомментирован, чтобы он не выполнялся при нажатии на "Run All".

In [5]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.bodyText"]]

# embeddings_full = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )

In [6]:
# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.trailText"]]

# embeddings_short = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )


Сохраняем наши эмбеддинги в файлы

In [7]:
# np.save("embeddings_f.npy", embeddings_full)

In [8]:
# np.save("embeddings_s.npy", embeddings_short)

Сразу же подгрузим и преобразуем в DataFrame

In [9]:
np_embeddings_full = np.load('embeddings_f.npy')
np_embeddings_short = np.load('embeddings_s.npy')

In [10]:
embeddings_full = pd.DataFrame(np_embeddings_full)
embeddings_short = pd.DataFrame(np_embeddings_short)

In [11]:
embeddings_full.shape

(5549, 384)

In [12]:

embeddings_short.shape

(7353, 384)

Соединим эмбеддинги с исходным датасетом, из которого оставим только то что пригодится для нашей модели - дата публикации, тикеры которые были указаны экспертами в статье, компании для которых эта новость была найдена, и источник новости.

In [13]:
embeddings_full.columns = [f"emb_f_{i}" for i in range(embeddings_full.shape[1])]
embeddings_short.columns = [f"emb_s_{i}" for i in range(embeddings_short.shape[1])]

In [14]:
df = df[["webPublicationDate", "stocks", "company", "source"]]

In [15]:
data = pd.concat(
    [
        df.reset_index(drop=True),
        embeddings_full.reset_index(drop=True),
        embeddings_short.reset_index(drop=True)
    ],
    axis=1
)

В столбце company для статей из FinHub есть пропуски, зато в тикерах пропусков нет. Восстановим компании по тикерам:

In [16]:
mapping = {
    "AAPL": "Apple",
    "TSLA": "Tesla",
    "CAT": "Caterpillar",
    "JPM": "JPMorgan",
    "NFLX": "Netflix",
    "PFE": "Pfizer",
    "XOM": "Exxon",
    "WMT": "Walmart"
}

In [17]:
data.loc[data["company"].isna(), "company"] = data.loc[data["company"].isna(), "stocks"].str.split(";").apply(lambda x: ";".join(mapping[i] for i in x if i in mapping))

In [18]:
data.isna().sum(axis=0)

webPublicationDate       0
stocks                4152
company                  0
source                   0
emb_f_0               1804
                      ... 
emb_s_379                0
emb_s_380                0
emb_s_381                0
emb_s_382                0
emb_s_383                0
Length: 772, dtype: int64

Для столбца stocks сделаем OHE оставив только нужные нам тикеры:

In [19]:
stocks = (data["stocks"].fillna("").str.get_dummies(sep=";"))[["AAPL", "TSLA", "CAT", "JPM", "NFLX", "PFE", "XOM", "WMT"]]

In [20]:
data = pd.concat([stocks, data], axis=1)

Аналогично сделаем OHE для столбца source:

In [21]:
source = data["source"].str.get_dummies()

In [22]:
data = pd.concat([source, data], axis=1)

In [23]:
data = data.drop(columns=["stocks", "source"])

Сохраним наш датасет под названием data_model

In [24]:
# data.to_csv("data_model.csv", index=False)

Разделим его на новости у которых есть полное содержимое и у которых нет:

In [33]:
common_cols = [c for c in data.columns if not c.startswith("emb_f_") and not c.startswith("emb_s_")]
f_cols = [c for c in data.columns if c.startswith("emb_f_")]
s_cols = [c for c in data.columns if c.startswith("emb_s_")]

data_f = data[common_cols + f_cols].copy()
data_s = data[common_cols + s_cols].copy()

In [34]:
data_f = data_f.dropna()
data_s = data_s.dropna()

In [ ]:
# data_f.to_csv("data_f.csv", index=False)
# data_s.to_csv("data_s.csv", index=False)

In [326]:
data_f = pd.read_csv("data_f.csv")
data_s = pd.read_csv("data_s.csv")

In [327]:
data_f["webPublicationDate"] = pd.to_datetime(data_f["webPublicationDate"])
data_s["webPublicationDate"] = pd.to_datetime(data_s["webPublicationDate"])

В столбце company у нас несколько компаний для одной и той же новости. Рзаделим их так, чтобы одна строка с несколькими компаниями превратилась в несколько разных строк, каждая с одной компанией. Так будет легче выделять нужные нам строки:

In [328]:
data_f["company"] = data_f["company"].str.split(";")
data_f = data_f.explode("company", ignore_index=True)

In [329]:
data_s["company"] = data_s["company"].str.split(";")
data_s = data_s.explode("company", ignore_index=True)

In [330]:
data_f.head()

,Benzinga,CNBC,ChartMill,DowJones,MarketWatch,SeekingAlpha,The Guardian,Yahoo,AAPL,TSLA,CAT,JPM,NFLX,PFE,XOM,WMT,webPublicationDate,company,emb_f_0,emb_f_1,emb_f_2,emb_f_3,emb_f_4,emb_f_5,emb_f_6,emb_f_7,emb_f_8,emb_f_9,emb_f_10,emb_f_11,emb_f_12,emb_f_13,emb_f_14,emb_f_15,emb_f_16,emb_f_17,emb_f_18,emb_f_19,emb_f_20,emb_f_21,emb_f_22,emb_f_23,emb_f_24,emb_f_25,emb_f_26,emb_f_27,emb_f_28,emb_f_29,emb_f_30,emb_f_31,emb_f_32,emb_f_33,emb_f_34,emb_f_35,emb_f_36,emb_f_37,emb_f_38,emb_f_39,emb_f_40,emb_f_41,emb_f_42,emb_f_43,emb_f_44,emb_f_45,emb_f_46,emb_f_47,emb_f_48,emb_f_49,emb_f_50,emb_f_51,emb_f_52,emb_f_53,emb_f_54,emb_f_55,emb_f_56,emb_f_57,emb_f_58,emb_f_59,emb_f_60,emb_f_61,emb_f_62,emb_f_63,emb_f_64,emb_f_65,emb_f_66,emb_f_67,emb_f_68,emb_f_69,emb_f_70,emb_f_71,emb_f_72,emb_f_73,emb_f_74,emb_f_75,emb_f_76,emb_f_77,emb_f_78,emb_f_79,emb_f_80,emb_f_81,emb_f_82,emb_f_83,emb_f_84,emb_f_85,emb_f_86,emb_f_87,emb_f_88,emb_f_89,emb_f_90,emb_f_91,emb_f_92,emb_f_93,emb_f_94,emb_f_95,emb_f_96,emb_f_97,emb_f_98,emb_f_99,emb_f_100,emb_f_101,emb_f_102,emb_f_103,emb_f_104,emb_f_105,emb_f_106,emb_f_107,emb_f_108,emb_f_109,emb_f_110,emb_f_111,emb_f_112,emb_f_113,emb_f_114,emb_f_115,emb_f_116,emb_f_117,emb_f_118,emb_f_119,emb_f_120,emb_f_121,emb_f_122,emb_f_123,emb_f_124,emb_f_125,emb_f_126,emb_f_127,emb_f_128,emb_f_129,emb_f_130,emb_f_131,emb_f_132,emb_f_133,emb_f_134,emb_f_135,emb_f_136,emb_f_137,emb_f_138,emb_f_139,emb_f_140,emb_f_141,emb_f_142,emb_f_143,emb_f_144,emb_f_145,emb_f_146,emb_f_147,emb_f_148,emb_f_149,emb_f_150,emb_f_151,emb_f_152,emb_f_153,emb_f_154,emb_f_155,emb_f_156,emb_f_157,emb_f_158,emb_f_159,emb_f_160,emb_f_161,emb_f_162,emb_f_163,emb_f_164,emb_f_165,emb_f_166,emb_f_167,emb_f_168,emb_f_169,emb_f_170,emb_f_171,emb_f_172,emb_f_173,emb_f_174,emb_f_175,emb_f_176,emb_f_177,emb_f_178,emb_f_179,emb_f_180,emb_f_181,emb_f_182,emb_f_183,emb_f_184,emb_f_185,emb_f_186,emb_f_187,emb_f_188,emb_f_189,emb_f_190,emb_f_191,emb_f_192,emb_f_193,emb_f_194,emb_f_195,emb_f_196,emb_f_197,emb_f_198,emb_f_199,emb_f_200,emb_f_201,emb_f_202,emb_f_203,emb_f_204,emb_f_205,emb_f_206,emb_f_207,emb_f_208,emb_f_209,emb_f_210,emb_f_211,emb_f_212,emb_f_213,emb_f_214,emb_f_215,emb_f_216,emb_f_217,emb_f_218,emb_f_219,emb_f_220,emb_f_221,emb_f_222,emb_f_223,emb_f_224,emb_f_225,emb_f_226,emb_f_227,emb_f_228,emb_f_229,emb_f_230,emb_f_231,emb_f_232,emb_f_233,emb_f_234,emb_f_235,emb_f_236,emb_f_237,emb_f_238,emb_f_239,emb_f_240,emb_f_241,emb_f_242,emb_f_243,emb_f_244,emb_f_245,emb_f_246,emb_f_247,emb_f_248,emb_f_249,emb_f_250,emb_f_251,emb_f_252,emb_f_253,emb_f_254,emb_f_255,emb_f_256,emb_f_257,emb_f_258,emb_f_259,emb_f_260,emb_f_261,emb_f_262,emb_f_263,emb_f_264,emb_f_265,emb_f_266,emb_f_267,emb_f_268,emb_f_269,emb_f_270,emb_f_271,emb_f_272,emb_f_273,emb_f_274,emb_f_275,emb_f_276,emb_f_277,emb_f_278,emb_f_279,emb_f_280,emb_f_281,emb_f_282,emb_f_283,emb_f_284,emb_f_285,emb_f_286,emb_f_287,emb_f_288,emb_f_289,emb_f_290,emb_f_291,emb_f_292,emb_f_293,emb_f_294,emb_f_295,emb_f_296,emb_f_297,emb_f_298,emb_f_299,emb_f_300,emb_f_301,emb_f_302,emb_f_303,emb_f_304,emb_f_305,emb_f_306,emb_f_307,emb_f_308,emb_f_309,emb_f_310,emb_f_311,emb_f_312,emb_f_313,emb_f_314,emb_f_315,emb_f_316,emb_f_317,emb_f_318,emb_f_319,emb_f_320,emb_f_321,emb_f_322,emb_f_323,emb_f_324,emb_f_325,emb_f_326,emb_f_327,emb_f_328,emb_f_329,emb_f_330,emb_f_331,emb_f_332,emb_f_333,emb_f_334,emb_f_335,emb_f_336,emb_f_337,emb_f_338,emb_f_339,emb_f_340,emb_f_341,emb_f_342,emb_f_343,emb_f_344,emb_f_345,emb_f_346,emb_f_347,emb_f_348,emb_f_349,emb_f_350,emb_f_351,emb_f_352,emb_f_353,emb_f_354,emb_f_355,emb_f_356,emb_f_357,emb_f_358,emb_f_359,emb_f_360,emb_f_361,emb_f_362,emb_f_363,emb_f_364,emb_f_365,emb_f_366,emb_f_367,emb_f_368,emb_f_369,emb_f_370,emb_f_371,emb_f_372,emb_f_373,emb_f_374,emb_f_375,emb_f_376,emb_f_377,emb_f_378,emb_f_379,emb_f_380,emb_f_381,emb_f_382,emb_f_383
0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,2026-03-23 13:31:06+00:00,Apple,0.038025,0.018858,-0.086595,0.008641,0.034031,-0.018835,0.076140

In [331]:
data_f["webPublicationDate"]

0      2026-03-23 13:31:06+00:00
1      2026-03-23 13:31:06+00:00
2      2026-03-23 13:31:06+00:00
3      2026-03-23 13:31:06+00:00
4      2025-07-23 10:56:43+00:00
                  ...           
8919   2025-04-22 08:42:26+00:00
8920   2025-07-23 18:40:37+00:00
8921   2025-02-10 06:17:04+00:00
8922   2025-03-28 05:00:12+00:00
8923   2025-04-17 17:21:57+00:00
Name: webPublicationDate, Length: 8924, dtype: datetime64[us, UTC]

Нам нужно вывести таргет для нашей модели. У нас в качестве таргета будет являться логарифм от процентного изменения цен. Изменение цен будем считать относительно последней цены, на которую текущая новость не повлияла - назовем такую цену P0. P1 - это цена закрытия в следующую торговую сессию. Pn - цена закрытия на n-ный торговую сессию после P0. А таргет соотвественно - log(Pn/P0) который мы будем считать для разных n с целью выявить лучший.

Чтобы получить последнюю цену закрытия, на которую текущая новость не повлияла, надо добавить само время закрытия в датасет котировок. В США это 20:00 для летнего времени и 21:00 для обычного:

In [332]:
quotes = pd.read_csv("quotes_all.csv", index_col=0)

In [333]:
quotes["Date"] = pd.to_datetime(quotes["Date"], utc=True)

In [334]:
quotes["is_summer"] = quotes["Date"].apply(lambda x: (x.month > 3 or (x.month == 3 and x.day >= 8)) and (x.month < 11 or (x.month == 11 and x.day < 1)))

In [335]:
quotes["Date"] = quotes["Date"] + pd.to_timedelta(np.where(quotes["is_summer"], "20:00:00", "21:00:00"))

In [336]:
quotes.head()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer
0,AAPL,2026-04-08 20:00:00+00:00,258.45,259.75,256.53,258.90,258.90,41008200.0,True
1,AAPL,2026-04-07 20:00:00+00:00,256.16,256.20,245.70,253.50,253.50,62148000.0,True
2,AAPL,2026-04-06 20:00:00+00:00,256.51,262.16,256.46,258.86,258.86,29329900.0,True
3,AAPL,2026-04-02 20:00:00+00:00,254.20,256.13,250.65,255.92,255.92,31289400.0,True
4,AAPL,2026-04-01 20:00:00+00:00,254.08,256.18,253.33,255.63,255.63,40059400.0,True


Приведем тикеры к названиям компаний

In [337]:
quotes["company"] = quotes["ticker"].map(mapping)

In [338]:
quotes.head()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer,company
0,AAPL,2026-04-08 20:00:00+00:00,258.45,259.75,256.53,258.90,258.90,41008200.0,True,Apple
1,AAPL,2026-04-07 20:00:00+00:00,256.16,256.20,245.70,253.50,253.50,62148000.0,True,Apple
2,AAPL,2026-04-06 20:00:00+00:00,256.51,262.16,256.46,258.86,258.86,29329900.0,True,Apple
3,AAPL,2026-04-02 20:00:00+00:00,254.20,256.13,250.65,255.92,255.92,31289400.0,True,Apple
4,AAPL,2026-04-01 20:00:00+00:00,254.08,256.18,253.33,255.63,255.63,40059400.0,True,Apple


Отсортируем все по дате

In [339]:
quotes = quotes.sort_values(["Date", "ticker"]).reset_index(drop=True)
data_f = data_f.sort_values(["webPublicationDate", "company"]).reset_index(drop=True)

Теперь создадим колонку с номером котировки для каждого тикера:

In [340]:
quotes["q_idx"] = quotes.groupby("ticker").cumcount()

In [341]:
quotes.head()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer,company,q_idx
0,AAPL,2020-12-31 21:00:00+00:00,134.08,134.74,131.72,132.69,129.05,99116600.0,False,Apple,0
1,CAT,2020-12-31 21:00:00+00:00,180.25,182.10,178.75,182.02,165.13,1717800.0,False,Caterpillar,0
2,JPM,2020-12-31 21:00:00+00:00,125.09,127.33,124.82,127.07,110.54,8580200.0,False,JPMorgan,0
3,NFLX,2020-12-31 21:00:00+00:00,52.55,54.55,52.31,54.07,54.07,53923000.0,False,Netflix,0
4,PFE,2020-12-31 21:00:00+00:00,36.66,36.92,36.29,36.81,28.37,30796500.0,False,Pfizer,0


Сначала найдем P0 - это самая поздняя цена закрытия на которую не повлияла новость. С этим нам поможет pd.merge_asof который делает мердж по ближайшему значению ключа. 

In [342]:
data_f = pd.merge_asof(
    data_f,
    quotes,
    left_on="webPublicationDate",
    right_on="Date",
    by="company",
    direction="backward"
)

In [343]:
data_f.head()

,Benzinga,CNBC,ChartMill,DowJones,MarketWatch,SeekingAlpha,The Guardian,Yahoo,AAPL,TSLA,CAT,JPM,NFLX,PFE,XOM,WMT,webPublicationDate,company,emb_f_0,emb_f_1,emb_f_2,emb_f_3,emb_f_4,emb_f_5,emb_f_6,emb_f_7,emb_f_8,emb_f_9,emb_f_10,emb_f_11,emb_f_12,emb_f_13,emb_f_14,emb_f_15,emb_f_16,emb_f_17,emb_f_18,emb_f_19,emb_f_20,emb_f_21,emb_f_22,emb_f_23,emb_f_24,emb_f_25,emb_f_26,emb_f_27,emb_f_28,emb_f_29,emb_f_30,emb_f_31,emb_f_32,emb_f_33,emb_f_34,emb_f_35,emb_f_36,emb_f_37,emb_f_38,emb_f_39,emb_f_40,emb_f_41,emb_f_42,emb_f_43,emb_f_44,emb_f_45,emb_f_46,emb_f_47,emb_f_48,emb_f_49,emb_f_50,emb_f_51,emb_f_52,emb_f_53,emb_f_54,emb_f_55,emb_f_56,emb_f_57,emb_f_58,emb_f_59,emb_f_60,emb_f_61,emb_f_62,emb_f_63,emb_f_64,emb_f_65,emb_f_66,emb_f_67,emb_f_68,emb_f_69,emb_f_70,emb_f_71,emb_f_72,emb_f_73,emb_f_74,emb_f_75,emb_f_76,emb_f_77,emb_f_78,emb_f_79,emb_f_80,emb_f_81,emb_f_82,emb_f_83,emb_f_84,emb_f_85,emb_f_86,emb_f_87,emb_f_88,emb_f_89,emb_f_90,emb_f_91,emb_f_92,emb_f_93,emb_f_94,emb_f_95,emb_f_96,emb_f_97,emb_f_98,emb_f_99,emb_f_100,emb_f_101,emb_f_102,emb_f_103,emb_f_104,emb_f_105,emb_f_106,emb_f_107,emb_f_108,emb_f_109,emb_f_110,emb_f_111,emb_f_112,emb_f_113,emb_f_114,emb_f_115,emb_f_116,emb_f_117,emb_f_118,emb_f_119,emb_f_120,emb_f_121,emb_f_122,emb_f_123,emb_f_124,emb_f_125,emb_f_126,emb_f_127,emb_f_128,emb_f_129,emb_f_130,emb_f_131,emb_f_132,emb_f_133,emb_f_134,emb_f_135,emb_f_136,emb_f_137,emb_f_138,emb_f_139,emb_f_140,emb_f_141,emb_f_142,emb_f_143,emb_f_144,emb_f_145,emb_f_146,emb_f_147,emb_f_148,emb_f_149,emb_f_150,emb_f_151,emb_f_152,emb_f_153,emb_f_154,emb_f_155,emb_f_156,emb_f_157,emb_f_158,emb_f_159,emb_f_160,emb_f_161,emb_f_162,emb_f_163,emb_f_164,emb_f_165,emb_f_166,emb_f_167,emb_f_168,emb_f_169,emb_f_170,emb_f_171,emb_f_172,emb_f_173,emb_f_174,emb_f_175,emb_f_176,emb_f_177,emb_f_178,emb_f_179,emb_f_180,emb_f_181,emb_f_182,emb_f_183,emb_f_184,emb_f_185,emb_f_186,emb_f_187,emb_f_188,emb_f_189,emb_f_190,emb_f_191,emb_f_192,emb_f_193,emb_f_194,emb_f_195,emb_f_196,emb_f_197,emb_f_198,emb_f_199,emb_f_200,emb_f_201,emb_f_202,emb_f_203,emb_f_204,emb_f_205,emb_f_206,emb_f_207,emb_f_208,emb_f_209,emb_f_210,emb_f_211,emb_f_212,emb_f_213,emb_f_214,emb_f_215,emb_f_216,emb_f_217,emb_f_218,emb_f_219,emb_f_220,emb_f_221,emb_f_222,emb_f_223,emb_f_224,emb_f_225,emb_f_226,emb_f_227,emb_f_228,emb_f_229,emb_f_230,emb_f_231,emb_f_232,emb_f_233,emb_f_234,emb_f_235,emb_f_236,emb_f_237,emb_f_238,emb_f_239,emb_f_240,emb_f_241,emb_f_242,emb_f_243,emb_f_244,emb_f_245,emb_f_246,emb_f_247,emb_f_248,emb_f_249,emb_f_250,emb_f_251,emb_f_252,emb_f_253,emb_f_254,emb_f_255,emb_f_256,emb_f_257,emb_f_258,emb_f_259,emb_f_260,emb_f_261,emb_f_262,emb_f_263,emb_f_264,emb_f_265,emb_f_266,emb_f_267,emb_f_268,emb_f_269,emb_f_270,emb_f_271,emb_f_272,emb_f_273,emb_f_274,emb_f_275,emb_f_276,emb_f_277,emb_f_278,emb_f_279,emb_f_280,emb_f_281,emb_f_282,emb_f_283,emb_f_284,emb_f_285,emb_f_286,emb_f_287,emb_f_288,emb_f_289,emb_f_290,emb_f_291,emb_f_292,emb_f_293,emb_f_294,emb_f_295,emb_f_296,emb_f_297,emb_f_298,emb_f_299,emb_f_300,emb_f_301,emb_f_302,emb_f_303,emb_f_304,emb_f_305,emb_f_306,emb_f_307,emb_f_308,emb_f_309,emb_f_310,emb_f_311,emb_f_312,emb_f_313,emb_f_314,emb_f_315,emb_f_316,emb_f_317,emb_f_318,emb_f_319,emb_f_320,emb_f_321,emb_f_322,emb_f_323,emb_f_324,emb_f_325,emb_f_326,emb_f_327,emb_f_328,emb_f_329,emb_f_330,emb_f_331,emb_f_332,emb_f_333,emb_f_334,emb_f_335,emb_f_336,emb_f_337,emb_f_338,emb_f_339,emb_f_340,emb_f_341,emb_f_342,emb_f_343,emb_f_344,emb_f_345,emb_f_346,emb_f_347,emb_f_348,emb_f_349,emb_f_350,emb_f_351,emb_f_352,emb_f_353,emb_f_354,emb_f_355,emb_f_356,emb_f_357,emb_f_358,emb_f_359,emb_f_360,emb_f_361,emb_f_362,emb_f_363,emb_f_364,emb_f_365,emb_f_366,emb_f_367,emb_f_368,emb_f_369,emb_f_370,emb_f_371,emb_f_372,emb_f_373,emb_f_374,emb_f_375,emb_f_376,emb_f_377,emb_f_378,emb_f_379,emb_f_380,emb_f_381,emb_f_382,emb_f_383,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer,q_idx
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2021-01-01 13:00:51+00:00,Pfize

Собственно Adj Close в самом конце это и есть наша P0. Как получить P1, P2 и следующие? Поскольку датафрейм уже отсортирован в рамках тикера, то мы пройтись по ним и построить матрицу будущих цен:

In [344]:
for k in range(31):
    quotes[f"P{k}"] = quotes.groupby("ticker")["Adj Close"].shift(-k)

In [345]:
quotes.head()

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer,company,q_idx,P0,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20,P21,P22,P23,P24,P25,P26,P27,P28,P29,P30
0,AAPL,2020-12-31 21:00:00+00:00,134.08,134.74,131.72,132.69,129.05,99116600.0,False,Apple,0,129.05,125.86,127.41,123.12,127.33,128.42,125.44,125.26,127.30,125.37,123.65,124.32,128.40,133.11,135.25,139.00,139.23,138.16,133.33,128.34,130.46,131.28,130.26,133.62,133.20,133.35,132.47,131.87,131.62,131.85,129.73
1,CAT,2020-12-31 21:00:00+00:00,180.25,182.10,178.75,182.02,165.13,1717800.0,False,Caterpillar,0,165.13,165.24,166.60,175.87,176.20,176.23,175.87,179.20,177.71,179.08,176.56,176.98,175.42,175.28,175.05,170.86,170.74,164.74,168.12,166.75,168.47,175.56,174.60,174.79,176.02,180.08,179.92,180.32,180.90,180.57,184.57
2,JPM,2020-12-31 21:00:00+00:00,125.09,127.33,124.82,127.07,110.54,8580200.0,False,JPMorgan,0,110.54,109.49,110.09,115.26,119.04,119.17,120.95,122.85,122.97,123.69,121.47,120.94,119.13,118.13,117.22,115.76,115.28,112.02,113.99,112.73,113.57,117.06,118.40,121.13,120.89,122.78,122.29,122.36,122.02,123.76,126.73
3,NFLX,2020-12-31 21:00:00+00:00,52.55,54.55,52.31,54.07,54.07,53923000.0,False,Netflix,0,54.07,52.29,52.08,50.05,50.89,51.04,49.91,49.42,50.78,50.09,49.80,50.18,58.63,57.98,56.52,55.68,56.19,52.33,53.86,53.24,53.90,54.82,53.94,55.22,55.08,54.79,55.91,56.36,55.76,55.65,55.73
4,PFE,2020-12-31 21:00:00+00:00,36.66,36.92,36.29,36.81,28.37,30796500.0,False,Pfizer,0,28.37,28.37,28.66,28.41,28.56,28.61,29.11,28.65,28.41,28.32,28.28,28.31,28.13,28.11,28.17,28.73,28.75,27.93,27.94,27.97,27.89,27.26,27.14,27.18,27.20,27.13,27.24,27.06,26.82,27.05,27.02


Осталось только присоединить эти значения к исходному датасету:

In [346]:
data_f = data_f.merge(
    quotes[["company", "q_idx"] + [f"P{k}" for k in range(31)]],
    on=["company", "q_idx"],
    how="left"
)

In [347]:
data_f.head()

,Benzinga,CNBC,ChartMill,DowJones,MarketWatch,SeekingAlpha,The Guardian,Yahoo,AAPL,TSLA,CAT,JPM,NFLX,PFE,XOM,WMT,webPublicationDate,company,emb_f_0,emb_f_1,emb_f_2,emb_f_3,emb_f_4,emb_f_5,emb_f_6,emb_f_7,emb_f_8,emb_f_9,emb_f_10,emb_f_11,emb_f_12,emb_f_13,emb_f_14,emb_f_15,emb_f_16,emb_f_17,emb_f_18,emb_f_19,emb_f_20,emb_f_21,emb_f_22,emb_f_23,emb_f_24,emb_f_25,emb_f_26,emb_f_27,emb_f_28,emb_f_29,emb_f_30,emb_f_31,emb_f_32,emb_f_33,emb_f_34,emb_f_35,emb_f_36,emb_f_37,emb_f_38,emb_f_39,emb_f_40,emb_f_41,emb_f_42,emb_f_43,emb_f_44,emb_f_45,emb_f_46,emb_f_47,emb_f_48,emb_f_49,emb_f_50,emb_f_51,emb_f_52,emb_f_53,emb_f_54,emb_f_55,emb_f_56,emb_f_57,emb_f_58,emb_f_59,emb_f_60,emb_f_61,emb_f_62,emb_f_63,emb_f_64,emb_f_65,emb_f_66,emb_f_67,emb_f_68,emb_f_69,emb_f_70,emb_f_71,emb_f_72,emb_f_73,emb_f_74,emb_f_75,emb_f_76,emb_f_77,emb_f_78,emb_f_79,emb_f_80,emb_f_81,emb_f_82,emb_f_83,emb_f_84,emb_f_85,emb_f_86,emb_f_87,emb_f_88,emb_f_89,emb_f_90,emb_f_91,emb_f_92,emb_f_93,emb_f_94,emb_f_95,emb_f_96,emb_f_97,emb_f_98,emb_f_99,emb_f_100,emb_f_101,emb_f_102,emb_f_103,emb_f_104,emb_f_105,emb_f_106,emb_f_107,emb_f_108,emb_f_109,emb_f_110,emb_f_111,emb_f_112,emb_f_113,emb_f_114,emb_f_115,emb_f_116,emb_f_117,emb_f_118,emb_f_119,emb_f_120,emb_f_121,emb_f_122,emb_f_123,emb_f_124,emb_f_125,emb_f_126,emb_f_127,emb_f_128,emb_f_129,emb_f_130,emb_f_131,emb_f_132,emb_f_133,emb_f_134,emb_f_135,emb_f_136,emb_f_137,emb_f_138,emb_f_139,emb_f_140,emb_f_141,emb_f_142,emb_f_143,emb_f_144,emb_f_145,emb_f_146,emb_f_147,emb_f_148,emb_f_149,emb_f_150,emb_f_151,emb_f_152,emb_f_153,emb_f_154,emb_f_155,emb_f_156,emb_f_157,emb_f_158,emb_f_159,emb_f_160,emb_f_161,emb_f_162,emb_f_163,emb_f_164,emb_f_165,emb_f_166,emb_f_167,emb_f_168,emb_f_169,emb_f_170,emb_f_171,emb_f_172,emb_f_173,emb_f_174,emb_f_175,emb_f_176,emb_f_177,emb_f_178,emb_f_179,emb_f_180,emb_f_181,emb_f_182,emb_f_183,emb_f_184,emb_f_185,emb_f_186,emb_f_187,emb_f_188,emb_f_189,emb_f_190,emb_f_191,emb_f_192,emb_f_193,emb_f_194,emb_f_195,emb_f_196,emb_f_197,emb_f_198,emb_f_199,emb_f_200,emb_f_201,emb_f_202,emb_f_203,emb_f_204,emb_f_205,emb_f_206,emb_f_207,emb_f_208,emb_f_209,emb_f_210,emb_f_211,emb_f_212,emb_f_213,emb_f_214,emb_f_215,emb_f_216,emb_f_217,emb_f_218,emb_f_219,emb_f_220,emb_f_221,emb_f_222,emb_f_223,emb_f_224,emb_f_225,emb_f_226,emb_f_227,emb_f_228,emb_f_229,emb_f_230,emb_f_231,emb_f_232,emb_f_233,emb_f_234,emb_f_235,emb_f_236,emb_f_237,emb_f_238,emb_f_239,emb_f_240,emb_f_241,emb_f_242,emb_f_243,emb_f_244,emb_f_245,emb_f_246,emb_f_247,emb_f_248,emb_f_249,emb_f_250,emb_f_251,emb_f_252,emb_f_253,emb_f_254,emb_f_255,emb_f_256,emb_f_257,emb_f_258,emb_f_259,emb_f_260,emb_f_261,emb_f_262,emb_f_263,emb_f_264,emb_f_265,emb_f_266,emb_f_267,emb_f_268,emb_f_269,emb_f_270,emb_f_271,emb_f_272,emb_f_273,emb_f_274,emb_f_275,emb_f_276,emb_f_277,emb_f_278,emb_f_279,emb_f_280,emb_f_281,emb_f_282,emb_f_283,emb_f_284,emb_f_285,emb_f_286,emb_f_287,emb_f_288,emb_f_289,emb_f_290,emb_f_291,emb_f_292,emb_f_293,emb_f_294,emb_f_295,emb_f_296,emb_f_297,emb_f_298,emb_f_299,emb_f_300,emb_f_301,emb_f_302,emb_f_303,emb_f_304,emb_f_305,emb_f_306,emb_f_307,emb_f_308,emb_f_309,emb_f_310,emb_f_311,emb_f_312,emb_f_313,emb_f_314,emb_f_315,emb_f_316,emb_f_317,emb_f_318,emb_f_319,emb_f_320,emb_f_321,emb_f_322,emb_f_323,emb_f_324,emb_f_325,emb_f_326,emb_f_327,emb_f_328,emb_f_329,emb_f_330,emb_f_331,emb_f_332,emb_f_333,emb_f_334,emb_f_335,emb_f_336,emb_f_337,emb_f_338,emb_f_339,emb_f_340,emb_f_341,emb_f_342,emb_f_343,emb_f_344,emb_f_345,emb_f_346,emb_f_347,emb_f_348,emb_f_349,emb_f_350,emb_f_351,emb_f_352,emb_f_353,emb_f_354,emb_f_355,emb_f_356,emb_f_357,emb_f_358,emb_f_359,emb_f_360,emb_f_361,emb_f_362,emb_f_363,emb_f_364,emb_f_365,emb_f_366,emb_f_367,emb_f_368,emb_f_369,emb_f_370,emb_f_371,emb_f_372,emb_f_373,emb_f_374,emb_f_375,emb_f_376,emb_f_377,emb_f_378,emb_f_379,emb_f_380,emb_f_381,emb_f_382,emb_f_383,ticker,Date,Open,High,Low,Close,Adj Close,Volume,is_summer,q_idx,P0,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,P16,P17,P18

In [348]:
data_f = data_f.dropna(axis=0)

Отлично, данные готовы, можем делить их на X и y:

In [349]:
y = data_f[["company"] + [f"P{k}" for k in range(31)]]
X = data_f.drop(columns=["ticker", "webPublicationDate", "Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "is_summer", "q_idx"] + [f"P{k}" for k in range(31)])

In [350]:
y.head()

,company,P0,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20,P21,P22,P23,P24,P25,P26,P27,P28,P29,P30
0,Pfizer,28.37,28.37,28.66,28.41,28.56,28.61,29.11,28.65,28.41,28.32,28.28,28.31,28.13,28.11,28.17,28.73,28.75,27.93,27.94,27.97,27.89,27.26,27.14,27.18,27.20,27.13,27.24,27.06,26.82,27.05,27.02
1,Netflix,54.07,52.29,52.08,50.05,50.89,51.04,49.91,49.42,50.78,50.09,49.80,50.18,58.63,57.98,56.52,55.68,56.19,52.33,53.86,53.24,53.90,54.82,53.94,55.22,55.08,54.79,55.91,56.36,55.76,55.65,55.73
2,Pfizer,28.37,28.37,28.66,28.41,28.56,28.61,29.11,28.65,28.41,28.32,28.28,28.31,28.13,28.11,28.17,28.73,28.75,27.93,27.94,27.97,27.89,27.26,27.14,27.18,27.20,27.13,27.24,27.06,26.82,27.05,27.02
3,JPMorgan,109.49,110.09,115.26,119.04,119.17,120.95,122.85,122.97,123.69,121.47,120.94,119.13,118.13,117.22,115.76,115.28,112.02,113.99,112.73,113.57,117.06,118.40,121.13,120.89,122.78,122.29,122.36,122.02,123.76,126.73,127.13
4,JPMorgan,109.49,110.09,115.26,119.04,119.17,120.95,122.85,122.97,123.69,121.47,120.94,119.13,118.13,117.22,115.76,115.28,112.02,113.99,112.73,113.57,117.06,118.40,121.13,120.89,122.78,122.29,122.36,122.02,123.76,126.73,127.13


Но не забудем что в y мы работаем именно с измнением цены:

In [351]:
y[[f"P{i}" for i in range(1, 31)]] = np.log(y[[f"P{i}" for i in range(1, 31)]].div(y["P0"], axis=0))

In [352]:
y.head()

,company,P0,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20,P21,P22,P23,P24,P25,P26,P27,P28,P29,P30
0,Pfizer,28.37,0.000000,0.010170,0.001409,0.006675,0.008424,0.025750,0.009821,0.001409,-0.001764,-0.003177,-0.002117,-0.008496,-0.009207,-0.007075,0.012610,0.013306,-0.015631,-0.015273,-0.014200,-0.017064,-0.039912,-0.044324,-0.042851,-0.042115,-0.044692,-0.040646,-0.047276,-0.056184,-0.047645,-0.048755
1,Netflix,54.07,-0.033474,-0.037499,-0.077257,-0.060613,-0.057670,-0.080058,-0.089924,-0.062777,-0.076458,-0.082265,-0.074663,0.080967,0.069819,0.044315,0.029342,0.038459,-0.032710,-0.003891,-0.015470,-0.003149,0.013776,-0.002407,0.021046,0.018507,0.013228,0.033464,0.041480,0.030777,0.028803,0.030239
2,Pfizer,28.37,0.000000,0.010170,0.001409,0.006675,0.008424,0.025750,0.009821,0.001409,-0.001764,-0.003177,-0.002117,-0.008496,-0.009207,-0.007075,0.012610,0.013306,-0.015631,-0.015273,-0.014200,-0.017064,-0.039912,-0.044324,-0.042851,-0.042115,-0.044692,-0.040646,-0.047276,-0.056184,-0.047645,-0.048755
3,JPMorgan,109.49,0.005465,0.051357,0.083626,0.084718,0.099544,0.115131,0.116107,0.121945,0.103834,0.099461,0.084382,0.075952,0.068219,0.055686,0.051531,0.022844,0.040278,0.029162,0.036586,0.066853,0.078236,0.101031,0.099048,0.114561,0.110562,0.111134,0.108352,0.122511,0.146226,0.149377
4,JPMorgan,109.49,0.005465,0.051357,0.083626,0.084718,0.099544,0.115131,0.116107,0.121945,0.103834,0.099461,0.084382,0.075952,0.068219,0.055686,0.051531,0.022844,0.040278,0.029162,0.036586,0.066853,0.078236,0.101031,0.099048,0.114561,0.110562,0.111134,0.108352,0.122511,0.146226,0.149377


Попробуем поработать с компанией Apple.

In [353]:
X_apple = X[X["company"] == "Apple"].drop(columns=["company"])
y_apple = y[y["company"] == "Apple"].drop(columns=["company", "P0"])

In [378]:
y_apple_bin = (y_apple[[f"P{i}" for i in range(1, 31)]] > 0).astype(int)

In [385]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

for i in range(1, 31):
    X_apple_train, X_apple_test, y_apple_train, y_apple_test = train_test_split(X_apple, y_apple_bin[f"P{i}"], test_size=0.2, random_state=67)

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_apple_train, y_apple_train)

    pred = model.predict(X_apple_test)
    prob = model.predict_proba(X_apple_test)[:, 1]

    print(f"F1 для P{i}: {f1_score(y_apple_test, pred)}")
    print(f"ROC-AUC для P{i}: {roc_auc_score(y_apple_test, prob)}")
    print()

F1 для P1: 0.6778242677824268
ROC-AUC для P1: 0.4483810076956585

F1 для P2: 0.6885245901639344
ROC-AUC для P2: 0.5123579545454545

F1 для P3: 0.7096774193548387
ROC-AUC для P3: 0.4743720052272397

F1 для P4: 0.701195219123506
ROC-AUC для P4: 0.4691447655002178

F1 для P5: 0.6831275720164609
ROC-AUC для P5: 0.468013468013468

F1 для P6: 0.712
ROC-AUC для P6: 0.5373892841585596

F1 для P7: 0.6987951807228916
ROC-AUC для P7: 0.5596209912536443

F1 для P8: 0.660377358490566
ROC-AUC для P8: 0.5501725129384704

F1 для P9: 0.6188340807174888
ROC-AUC для P9: 0.43243243243243246

F1 для P10: 0.6829268292682927
ROC-AUC для P10: 0.5037682872764888

F1 для P11: 0.7258687258687259
ROC-AUC для P11: 0.4949494949494949

F1 для P12: 0.7518796992481203
ROC-AUC для P12: 0.5393801965230537

F1 для P13: 0.7715355805243446
ROC-AUC для P13: 0.5479938271604939

F1 для P14: 0.7777777777777778
ROC-AUC для P14: 0.5179598818224227

F1 для P15: 0.781021897810219
ROC-AUC для P15: 0.4858280986670752

F1 для P16: 0.

Наилучший F1 при задержке в 15 дней. Однако ROC-AUC оставляет желать лучшего - прогноз как у случайного угадывания.

Вывод: Модель в текущем виде не пригодна для практического использования. Нужно возвращаться к анализу данных и выбору алгоритма.